<a href="https://colab.research.google.com/github/njwbilll/Tugas-1_Introduction-to-Machine-Learning-with-Python-O-Reilly-_Najwa-Bilqis-Al-Khalidah/blob/main/02_Supervised_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 2: Supervised Learning

**Referensi:** Introduction to Machine Learning with Python -- Andreas C. Muller & Sarah Guido (O'Reilly)

---

## Ringkasan Chapter

Supervised learning adalah cabang machine learning yang paling banyak digunakan.
Pada chapter ini kita mempelajari berbagai algoritma supervised learning mulai dari
yang paling sederhana hingga yang kompleks, beserta cara memilih algoritma yang tepat
untuk suatu masalah.

**Algoritma yang dibahas:**
- k-Nearest Neighbors (kNN) untuk klasifikasi dan regresi
- Linear Models: Ridge, Lasso, Logistic Regression, Linear SVM
- Naive Bayes Classifiers
- Decision Trees
- Ensemble Methods: Random Forest dan Gradient Boosted Trees
- Kernelized Support Vector Machines (SVM)
- Neural Networks (Multilayer Perceptrons)
- Estimasi ketidakpastian dari classifier


## 2.0 Import Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import (
    load_breast_cancer, load_iris, make_moons,
    make_blobs, make_classification, make_regression
)
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
print("Library berhasil diimport.")


## 2.1 Klasifikasi vs. Regresi

Supervised learning terbagi menjadi dua jenis utama:

**Klasifikasi (Classification)**
- Memprediksi label kelas yang diskret (terbatas)
- Output berupa kategori: spam/bukan spam, sakit/sehat, jenis bunga, dll
- Contoh algoritma: kNN, Logistic Regression, Decision Tree, SVM

**Regresi (Regression)**
- Memprediksi nilai kontinu pada skala yang berkelanjutan
- Output berupa angka real: harga rumah, suhu, pendapatan, dll
- Contoh algoritma: Linear Regression, Ridge, Lasso, SVR

**Perbedaan mendasar:** pada klasifikasi kita prediksi "kelas mana",
sedangkan pada regresi kita prediksi "berapa nilainya".


In [ ]:
# Demonstrasi perbedaan output klasifikasi vs regresi
cancer = load_breast_cancer()

print("=== Contoh Dataset Klasifikasi (Breast Cancer) ===")
print(f"Jumlah sampel  : {cancer.data.shape[0]}")
print(f"Jumlah fitur   : {cancer.data.shape[1]}")
print(f"Kelas target   : {cancer.target_names}")
print(f"5 label pertama: {cancer.target[:5]}")
print()

X_reg, y_reg = make_regression(n_samples=100, n_features=1, noise=20, random_state=42)
print("=== Contoh Dataset Regresi ===")
print(f"Jumlah sampel  : {X_reg.shape[0]}")
print(f"Jumlah fitur   : {X_reg.shape[1]}")
print(f"5 nilai pertama: {y_reg[:5].round(2)}")
print("Target berupa nilai kontinu, bukan kategori.")


In [ ]:
# Visualisasi perbedaan klasifikasi vs regresi
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Klasifikasi: scatter plot dengan warna per kelas
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_2d = pca.fit_transform(cancer.data)
colors = ["steelblue", "tomato"]
for cls, color, name in zip([0, 1], colors, cancer.target_names):
    mask = cancer.target == cls
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color,
                    label=name, edgecolors="k", s=20, alpha=0.6)
axes[0].set_title("Klasifikasi: Breast Cancer (PCA 2D)")
axes[0].set_xlabel("Komponen Utama 1")
axes[0].set_ylabel("Komponen Utama 2")
axes[0].legend()

# Regresi: scatter plot dengan garis regresi
from sklearn.linear_model import LinearRegression
lr = LinearRegression().fit(X_reg, y_reg)
X_plot = np.linspace(X_reg.min(), X_reg.max(), 100).reshape(-1, 1)
axes[1].scatter(X_reg, y_reg, color="steelblue", edgecolors="k", s=30, alpha=0.7, label="Data")
axes[1].plot(X_plot, lr.predict(X_plot), color="tomato", linewidth=2, label="Model Linear")
axes[1].set_title("Regresi: Prediksi Nilai Kontinu")
axes[1].set_xlabel("Fitur X")
axes[1].set_ylabel("Target y")
axes[1].legend()

plt.tight_layout()
plt.show()


## 2.2 Generalisasi, Overfitting, dan Underfitting

Ini adalah konsep paling penting dalam machine learning.

**Generalisasi**
Kemampuan model untuk bekerja dengan baik pada data baru yang belum pernah dilihat
saat training. Inilah tujuan utama kita membangun model ML.

**Overfitting**
Model terlalu "hafal" data training hingga menangkap noise dan keacakan,
bukan pola yang sesungguhnya.
- Ciri: akurasi training sangat tinggi, akurasi test rendah
- Penyebab: model terlalu kompleks relatif terhadap jumlah data

**Underfitting**
Model terlalu sederhana sehingga tidak mampu menangkap pola yang ada di data.
- Ciri: akurasi training dan test sama-sama rendah
- Penyebab: model tidak cukup kompleks

**Trade-off model complexity vs generalization**
Kita perlu menemukan titik tengah: model yang cukup kompleks untuk menangkap
pola nyata, namun tidak terlalu kompleks sehingga menghafal noise.


In [ ]:
# Visualisasi overfitting vs underfitting pada regresi polinomial
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

np.random.seed(42)
X_toy = np.sort(np.random.rand(30))
y_toy = np.sin(2 * np.pi * X_toy) + np.random.randn(30) * 0.3
X_plot = np.linspace(0, 1, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
configs = [
    (1,  "Underfitting
(degree=1, terlalu sederhana)"),
    (4,  "Pas
(degree=4, generalisasi baik)"),
    (15, "Overfitting
(degree=15, terlalu kompleks)"),
]

for ax, (deg, title) in zip(axes, configs):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_toy.reshape(-1, 1), y_toy)

    train_score = model.score(X_toy.reshape(-1, 1), y_toy)
    y_hat = model.predict(X_plot.reshape(-1, 1))

    ax.scatter(X_toy, y_toy, color="steelblue", s=30, zorder=5, label="Data training")
    ax.plot(X_plot, y_hat, color="tomato", linewidth=2, label="Model")
    ax.plot(X_plot, np.sin(2 * np.pi * X_plot), color="gray",
            linewidth=1, linestyle="--", label="Fungsi asli")
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(f"{title}
Train R2={train_score:.3f}", fontsize=10)
    ax.legend(fontsize=7)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.tight_layout()
plt.show()
print("Degree 1 terlalu kaku, degree 15 mengikuti noise, degree 4 sesuai pola aslinya.")


## 2.3 k-Nearest Neighbors (kNN)

kNN adalah algoritma non-parametrik yang sangat intuitif.

**Cara kerja kNN untuk klasifikasi:**
1. Simpan semua data training.
2. Untuk data baru, hitung jarak ke semua titik training (biasanya Euclidean distance).
3. Ambil k titik training terdekat (k tetangga terdekat).
4. Prediksi kelas = kelas mayoritas dari k tetangga tersebut.

**Cara kerja kNN untuk regresi:**
Sama seperti klasifikasi, tapi prediksi = rata-rata nilai target dari k tetangga.

**Pengaruh nilai k:**
- k kecil (k=1): batas keputusan sangat kompleks, rentan overfitting
- k besar: batas keputusan lebih halus, tapi bisa underfitting

**Kelebihan:** sederhana, mudah dipahami, tidak ada asumsi tentang distribusi data.

**Kekurangan:** lambat saat prediksi pada dataset besar (harus hitung jarak ke semua titik training),
sensitif terhadap fitur yang tidak relevan dan skala fitur.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Dataset moons untuk visualisasi batas keputusan
X_moons, y_moons = make_moons(n_samples=150, noise=0.25, random_state=3)
X_mtr, X_mte, y_mtr, y_mte = train_test_split(X_moons, y_moons,
                                                test_size=0.3, random_state=0)

def plot_decision_boundary(clf, X, y, ax, title):
    h = 0.02
    x0_min, x0_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x1_min, x1_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x0_min, x0_max, h),
                          np.arange(x1_min, x1_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3,
                cmap=ListedColormap(["#AAAAFF", "#FFAAAA"]))
    ax.scatter(X[:, 0], X[:, 1], c=y,
               cmap=ListedColormap(["steelblue", "tomato"]),
               edgecolors="k", s=30)
    ax.set_title(title, fontsize=10)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
k_values = [1, 5, 20]

for k, ax in zip(k_values, axes):
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_mtr, y_mtr)
    train_acc = clf.score(X_mtr, y_mtr)
    test_acc  = clf.score(X_mte, y_mte)
    plot_decision_boundary(clf, X_moons, y_moons, ax,
        f"kNN k={k}
Train={train_acc:.2f}  Test={test_acc:.2f}")

plt.suptitle("Pengaruh Nilai k terhadap Batas Keputusan kNN", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# kNN untuk Regresi
from sklearn.neighbors import KNeighborsRegressor

np.random.seed(0)
X_knn_reg = np.sort(np.random.uniform(0, 10, 40))
y_knn_reg = np.sin(X_knn_reg) + np.random.randn(40) * 0.3

X_plot_r = np.linspace(0, 10, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for k, ax in zip([1, 3, 10], axes):
    reg = KNeighborsRegressor(n_neighbors=k)
    reg.fit(X_knn_reg.reshape(-1, 1), y_knn_reg)
    ax.scatter(X_knn_reg, y_knn_reg, color="steelblue", edgecolors="k", s=30, label="Data")
    ax.plot(X_plot_r, reg.predict(X_plot_r), color="tomato", linewidth=2, label=f"k={k}")
    ax.set_title(f"kNN Regresi k={k}")
    ax.legend()
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.tight_layout()
plt.show()
print("k=1 mengikuti setiap titik data (overfitting).")
print("k=10 terlalu halus dan melewatkan pola lokal (underfitting).")


## 2.4 Linear Models

Linear model memprediksi output menggunakan fungsi linear dari fitur input:

    y_hat = w[0]*x[0] + w[1]*x[1] + ... + w[p]*x[p] + b

di mana:
- `w` = bobot (weights / koefisien) yang dipelajari dari data
- `b` = bias (intercept)
- `p` = jumlah fitur

**Regularisasi** menambahkan penalti pada loss function untuk mencegah bobot menjadi
terlalu besar, sehingga mengurangi overfitting.

### 2.4.1 Ridge Regression (Regularisasi L2)

Menambahkan penalti berupa jumlah kuadrat semua bobot:
loss = MSE + alpha * sum(w^2)

Parameter `alpha` mengontrol kekuatan regularisasi:
- alpha kecil = sedikit regularisasi (mendekati Linear Regression biasa)
- alpha besar = regularisasi kuat (bobot mendekati nol)

### 2.4.2 Lasso Regression (Regularisasi L1)

Menambahkan penalti berupa jumlah nilai absolut semua bobot:
loss = MSE + alpha * sum(|w|)

Keunggulan Lasso: secara otomatis melakukan feature selection karena
beberapa bobot akan menjadi persis nol (sparse solution).


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Dataset regresi dengan banyak fitur (sebagian tidak informatif)
X_r, y_r = make_regression(n_samples=80, n_features=40, n_informative=8,
                             noise=30, random_state=0)
X_rtr, X_rte, y_rtr, y_rte = train_test_split(X_r, y_r, random_state=0)

# Bandingkan tiga model
models = [
    ("Linear Regression", LinearRegression()),
    ("Ridge (alpha=1)",   Ridge(alpha=1)),
    ("Ridge (alpha=10)",  Ridge(alpha=10)),
    ("Lasso (alpha=1)",   Lasso(alpha=1)),
    ("Lasso (alpha=3)",   Lasso(alpha=3)),
]

print(f"{'Model':<22} {'Train R2':>10} {'Test R2':>10} {'Non-zero coef':>15}")
print("-" * 60)
for name, m in models:
    m.fit(X_rtr, y_rtr)
    nonzero = np.sum(m.coef_ != 0) if hasattr(m, "coef_") else "N/A"
    print(f"{name:<22} {m.score(X_rtr, y_rtr):>10.3f} "
          f"{m.score(X_rte, y_rte):>10.3f} {str(nonzero):>15}")

print()
print("Lasso memaksa banyak koefisien menjadi nol = seleksi fitur otomatis.")


In [ ]:
# Visualisasi koefisien Ridge vs Lasso
ridge1 = Ridge(alpha=1).fit(X_rtr, y_rtr)
lasso1 = Lasso(alpha=1).fit(X_rtr, y_rtr)
lr_base = LinearRegression().fit(X_rtr, y_rtr)

plt.figure(figsize=(12, 5))
plt.plot(lr_base.coef_, "o", label="Linear Regression", alpha=0.7, color="gray")
plt.plot(ridge1.coef_,  "s", label="Ridge (alpha=1)",   alpha=0.8, color="steelblue")
plt.plot(lasso1.coef_,  "^", label="Lasso (alpha=1)",   alpha=0.8, color="tomato")
plt.axhline(0, color="black", linewidth=0.8, linestyle="--")
plt.xlabel("Indeks Fitur")
plt.ylabel("Nilai Koefisien")
plt.title("Perbandingan Koefisien: Linear Regression vs Ridge vs Lasso")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

lasso_zero = np.sum(lasso1.coef_ == 0)
print(f"Lasso: {lasso_zero} dari {len(lasso1.coef_)} koefisien bernilai tepat nol.")
print("Fitur dengan koefisien nol = tidak digunakan oleh model Lasso.")


### 2.4.3 Logistic Regression untuk Klasifikasi

Meskipun namanya mengandung "Regression", Logistic Regression adalah algoritma klasifikasi.
Ia menggunakan fungsi sigmoid untuk mengubah output linear menjadi probabilitas kelas:

    P(y=1) = sigmoid(w*x + b) = 1 / (1 + exp(-(w*x + b)))

Parameter `C` mengontrol regularisasi (kebalikan dari alpha):
- C besar = sedikit regularisasi = model lebih kompleks
- C kecil = regularisasi kuat = model lebih sederhana

Logistic Regression menggunakan regularisasi L2 secara default.


In [ ]:
from sklearn.linear_model import LogisticRegression

cancer = load_breast_cancer()
X_ctr, X_cte, y_ctr, y_cte = train_test_split(
    cancer.data, cancer.target, random_state=42, stratify=cancer.target)

print("Pengaruh C pada Logistic Regression (Breast Cancer):")
print(f"{'C':>8} {'Train Acc':>12} {'Test Acc':>12}")
print("-" * 35)
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    clf = LogisticRegression(max_iter=5000, C=C, random_state=0)
    clf.fit(X_ctr, y_ctr)
    print(f"{C:>8} {clf.score(X_ctr, y_ctr):>12.4f} {clf.score(X_cte, y_cte):>12.4f}")

print()
print("C kecil: regularisasi kuat, model lebih sederhana.")
print("C besar: regularisasi lemah, model lebih kompleks (risiko overfitting).")


In [ ]:
# Visualisasi koefisien Logistic Regression pada Cancer dataset
lr_small = LogisticRegression(max_iter=5000, C=0.01).fit(X_ctr, y_ctr)
lr_large = LogisticRegression(max_iter=5000, C=100).fit(X_ctr, y_ctr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model, title in zip(axes,
                              [lr_small, lr_large],
                              ["Logistic Regression C=0.01
(regularisasi kuat)",
                               "Logistic Regression C=100
(regularisasi lemah)"]):
    coef_series = pd.Series(model.coef_[0], index=cancer.feature_names)
    coef_series.plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel("Nilai Koefisien")
    ax.tick_params(axis="x", labelsize=7, rotation=60)

plt.tight_layout()
plt.show()
print("C kecil: koefisien lebih kecil dan merata.")
print("C besar: koefisien bisa sangat besar untuk fitur-fitur tertentu.")


## 2.5 Naive Bayes Classifiers

Naive Bayes menerapkan teorema Bayes dengan asumsi "naive" bahwa semua fitur
bersifat independen satu sama lain jika diberikan kelas tertentu.

**Teorema Bayes:**

    P(kelas | fitur) = P(fitur | kelas) * P(kelas) / P(fitur)

**Asumsi naive (dan tidak realistis):**
Dalam kenyataannya hampir tidak ada fitur yang benar-benar independen,
namun Naive Bayes tetap bekerja sangat baik dalam praktik.

**Tiga varian Naive Bayes:**

1. **GaussianNB**: Untuk fitur kontinu. Mengasumsikan setiap fitur mengikuti
   distribusi Gaussian (normal) pada setiap kelas. Menyimpan rata-rata dan
   varians per fitur per kelas.

2. **MultinomialNB**: Untuk data hitungan (count data), sangat umum digunakan
   pada klasifikasi teks (jumlah kemunculan kata).

3. **BernoulliNB**: Untuk fitur biner (0 atau 1). Cocok untuk data teks
   di mana kita hanya peduli apakah kata muncul atau tidak.

**Kelebihan:** sangat cepat saat training dan prediksi, bekerja baik
dengan dataset kecil, efektif untuk data berdimensi tinggi.

**Kekurangan:** asumsi independensi hampir selalu dilanggar,
estimasi probabilitas biasanya tidak akurat.


In [ ]:
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# GaussianNB pada Breast Cancer
gnb = GaussianNB()
gnb.fit(X_ctr, y_ctr)

print("=== Gaussian Naive Bayes pada Breast Cancer ===")
print(f"Training accuracy : {gnb.score(X_ctr, y_ctr):.4f}")
print(f"Test accuracy     : {gnb.score(X_cte, y_cte):.4f}")
print()
print(f"Kelas yang dipelajari : {gnb.classes_}")
print(f"Jumlah kelas          : {len(gnb.classes_)}")
print()
print("Parameter yang disimpan GaussianNB:")
print(f"  theta_ (rata-rata per kelas per fitur) shape: {gnb.theta_.shape}")
print(f"  var_ (varians per kelas per fitur)    shape: {gnb.var_.shape}")
print()
print("GaussianNB menyimpan rata-rata dan varians setiap fitur")
print("untuk setiap kelas, bukan model yang belajar bobot secara langsung.")


In [ ]:
# Visualisasi distribusi Gaussian yang dipelajari GaussianNB (2 fitur pertama)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
feat_names = cancer.feature_names
colors_cls = ["steelblue", "tomato"]
class_names = cancer.target_names

for row, feat_idx in enumerate([0, 1]):
    for col, cls in enumerate([0, 1]):
        ax = axes[row, col]
        # Histogram data aktual
        mask = y_ctr == cls
        ax.hist(X_ctr[mask, feat_idx], bins=25, alpha=0.5,
                color=colors_cls[col], density=True,
                label=f"Data {class_names[cls]}", edgecolor="none")
        # Kurva Gaussian yang dipelajari
        mu  = gnb.theta_[cls, feat_idx]
        std = np.sqrt(gnb.var_[cls, feat_idx])
        x_range = np.linspace(X_ctr[:, feat_idx].min(), X_ctr[:, feat_idx].max(), 200)
        from scipy.stats import norm
        ax.plot(x_range, norm.pdf(x_range, mu, std),
                color="black", linewidth=2, label="Distribusi Gaussian")
        ax.set_title(f"{class_names[cls]} | {feat_names[feat_idx][:25]}", fontsize=9)
        ax.set_xlabel("Nilai Fitur")
        ax.legend(fontsize=7)

plt.suptitle("Distribusi Gaussian yang Dipelajari GaussianNB per Kelas", fontsize=11)
plt.tight_layout()
plt.show()


## 2.6 Decision Trees

Decision tree membangun model dalam bentuk diagram alir (flowchart).
Setiap node internal menguji satu fitur, setiap cabang mewakili hasil pengujian,
dan setiap daun (leaf) memberikan prediksi.

**Cara membangun tree:**
Pada setiap node, algoritma memilih fitur dan nilai threshold yang paling baik
memisahkan kelas, menggunakan ukuran seperti Gini impurity atau information gain.

**Gini Impurity:**

    Gini = 1 - sum(p_i^2)

di mana p_i adalah proporsi kelas i. Nilai 0 = node murni (satu kelas saja).

**Hiperparameter penting:**
- `max_depth`: kedalaman maksimum tree. Senjata utama melawan overfitting.
- `min_samples_leaf`: minimum sampel yang harus ada di setiap daun.
- `min_samples_split`: minimum sampel untuk melakukan split.

**Kelebihan:**
- Sangat mudah divisualisasi dan diinterpretasi
- Tidak memerlukan normalisasi fitur
- Bisa menangani fitur numerik maupun kategorikal

**Kekurangan:**
- Sangat rentan overfitting jika tidak dipangkas
- Tidak stabil: perubahan kecil pada data bisa menghasilkan tree yang sangat berbeda


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Tree tanpa batasan (overfitting)
dt_full = DecisionTreeClassifier(random_state=0)
dt_full.fit(X_ctr, y_ctr)

# Tree dengan max_depth=4 (dipangkas)
dt_pruned = DecisionTreeClassifier(max_depth=4, random_state=0)
dt_pruned.fit(X_ctr, y_ctr)

print("=== Perbandingan Decision Tree ===")
print(f"{'Model':<30} {'Kedalaman':>10} {'Train Acc':>10} {'Test Acc':>10}")
print("-" * 62)
for name, model in [("Tanpa batas (overfitting)", dt_full),
                     ("max_depth=4 (dipangkas)",   dt_pruned)]:
    print(f"{name:<30} {model.get_depth():>10} "
          f"{model.score(X_ctr, y_ctr):>10.4f} "
          f"{model.score(X_cte, y_cte):>10.4f}")

print()
print("Tree tanpa batas: training sempurna tapi test buruk = overfitting.")
print("Tree dipangkas  : sedikit turun di training tapi test meningkat.")


In [ ]:
# Visualisasi tree yang dipangkas
plt.figure(figsize=(20, 8))
plot_tree(dt_pruned, feature_names=cancer.feature_names,
          class_names=cancer.target_names, filled=True,
          rounded=True, fontsize=8, max_depth=3)
plt.title("Struktur Decision Tree (max_depth=4, ditampilkan 3 level)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Feature Importance dari Decision Tree
feat_imp = pd.Series(dt_pruned.feature_importances_,
                      index=cancer.feature_names).sort_values(ascending=False)

print("Top 10 fitur terpenting menurut Decision Tree:")
print(feat_imp.head(10).round(4).to_string())
print()
print("Fitur dengan importance=0 tidak pernah digunakan untuk split.")

plt.figure(figsize=(10, 5))
feat_imp.head(15).sort_values().plot(kind="barh", color="steelblue", edgecolor="black")
plt.xlabel("Feature Importance (Gini)")
plt.title("Top 15 Feature Importance -- Decision Tree (max_depth=4)")
plt.tight_layout()
plt.show()


In [ ]:
# Pengaruh max_depth terhadap akurasi
depths = range(1, 16)
train_scores, test_scores = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=0)
    dt.fit(X_ctr, y_ctr)
    train_scores.append(dt.score(X_ctr, y_ctr))
    test_scores.append(dt.score(X_cte, y_cte))

plt.figure(figsize=(9, 5))
plt.plot(depths, train_scores, "o-", label="Training accuracy", color="steelblue")
plt.plot(depths, test_scores,  "s-", label="Test accuracy",     color="tomato")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Pengaruh max_depth terhadap Akurasi Decision Tree")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_depth = list(depths)[np.argmax(test_scores)]
print(f"Kedalaman optimal: {best_depth} dengan test accuracy: {max(test_scores):.4f}")


## 2.7 Ensemble Methods: Random Forest dan Gradient Boosted Trees

Ensemble method menggabungkan banyak model lemah menjadi satu model kuat.
Dua pendekatan utama adalah bagging (Random Forest) dan boosting (Gradient Boosted Trees).

### 2.7.1 Random Forest

Random Forest membangun banyak decision tree secara paralel dengan dua sumber keacakan:

1. **Bootstrap sampling**: setiap tree dilatih pada subset acak data training
   (dengan pengembalian, artinya satu sampel bisa muncul lebih dari sekali).

2. **Random feature selection**: pada setiap split, hanya subset acak fitur
   yang dipertimbangkan (defaultnya sqrt(n_features) untuk klasifikasi).

Prediksi akhir = mayoritas suara dari semua tree (klasifikasi)
atau rata-rata prediksi semua tree (regresi).

**Mengapa ini berhasil?**
Karena tree-tree yang berbeda membuat kesalahan yang berbeda-beda,
dan ketika kita rata-ratakan mereka, kesalahan individual saling mengkompensasi.

**Hiperparameter penting:**
- `n_estimators`: jumlah tree. Semakin banyak biasanya semakin baik,
  tapi semakin lama training.
- `max_features`: jumlah fitur yang dipertimbangkan di setiap split.
- `max_depth`: kedalaman setiap tree.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
rf.fit(X_ctr, y_ctr)

print("=== Random Forest (100 trees) pada Breast Cancer ===")
print(f"Training accuracy : {rf.score(X_ctr, y_ctr):.4f}")
print(f"Test accuracy     : {rf.score(X_cte, y_cte):.4f}")
print()

# Pengaruh n_estimators
print("Pengaruh jumlah tree (n_estimators):")
print(f"{'n_estimators':>15} {'Train Acc':>12} {'Test Acc':>10}")
print("-" * 40)
for n in [1, 5, 10, 50, 100, 200]:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=0, n_jobs=-1)
    rf_n.fit(X_ctr, y_ctr)
    print(f"{n:>15} {rf_n.score(X_ctr, y_ctr):>12.4f} {rf_n.score(X_cte, y_cte):>10.4f}")

print()
print("Setelah cukup banyak tree, performa mulai stabil (diminishing returns).")


In [ ]:
# Feature importance Random Forest vs Decision Tree
rf_imp = pd.Series(rf.feature_importances_, index=cancer.feature_names)
dt_imp = pd.Series(dt_pruned.feature_importances_, index=cancer.feature_names)

# Ambil top 15 dari Random Forest
top15_feats = rf_imp.nlargest(15).index

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
rf_imp[top15_feats].sort_values().plot(
    kind="barh", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Feature Importance: Random Forest")
axes[0].set_xlabel("Importance")

dt_imp[top15_feats].sort_values().plot(
    kind="barh", ax=axes[1], color="tomato", edgecolor="black")
axes[1].set_title("Feature Importance: Decision Tree (max_depth=4)")
axes[1].set_xlabel("Importance")

plt.suptitle("Perbandingan Feature Importance: RF vs Single Tree", fontsize=11)
plt.tight_layout()
plt.show()
print("Random Forest menggunakan lebih banyak fitur secara merata.")
print("Single tree hanya fokus pada beberapa fitur paling diskriminatif.")


### 2.7.2 Gradient Boosted Trees

Gradient Boosted Trees membangun tree secara **sekuensial** (satu per satu),
di mana setiap tree baru mencoba memperbaiki kesalahan prediksi ensemble sebelumnya.

**Perbedaan utama dengan Random Forest:**
- Random Forest: tree dibangun paralel dan independen
- Gradient Boosting: tree dibangun sekuensial, setiap tree "belajar dari kesalahan"

**Cara kerja:**
1. Mulai dengan prediksi awal (biasanya rata-rata target).
2. Hitung residual error (selisih prediksi vs aktual).
3. Latih tree baru untuk memprediksi residual error tersebut.
4. Tambahkan prediksi tree baru ke ensemble dengan learning rate tertentu.
5. Ulangi langkah 2-4 sebanyak n_estimators kali.

**Hiperparameter penting:**
- `n_estimators`: jumlah tree. Berbeda dengan RF, terlalu banyak bisa overfitting.
- `learning_rate`: seberapa besar kontribusi setiap tree. Harus diimbangi dengan n_estimators.
- `max_depth`: kedalaman setiap tree. Biasanya tree dangkal (3-5) bekerja paling baik.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
gb.fit(X_ctr, y_ctr)

print("=== Gradient Boosted Trees pada Breast Cancer ===")
print(f"Training accuracy : {gb.score(X_ctr, y_ctr):.4f}")
print(f"Test accuracy     : {gb.score(X_cte, y_cte):.4f}")
print()

# Pengaruh learning_rate dan n_estimators
print("Pengaruh learning_rate (n_estimators=100):")
print(f"{'learning_rate':>15} {'Train Acc':>12} {'Test Acc':>10}")
print("-" * 40)
for lr in [0.001, 0.01, 0.1, 0.3, 1.0]:
    gb_lr = GradientBoostingClassifier(n_estimators=100, learning_rate=lr,
                                         max_depth=3, random_state=0)
    gb_lr.fit(X_ctr, y_ctr)
    print(f"{lr:>15} {gb_lr.score(X_ctr, y_ctr):>12.4f} {gb_lr.score(X_cte, y_cte):>10.4f}")

print()
print("Learning rate terlalu besar atau kecil sama-sama merugikan performa.")


In [ ]:
# Perbandingan akhir: semua tree-based model
from sklearn.tree import DecisionTreeClassifier

models_tree = {
    "Decision Tree (max_depth=4)": DecisionTreeClassifier(max_depth=4, random_state=0),
    "Random Forest (100 trees)":   RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1),
    "Gradient Boosting (lr=0.1)":  GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                                max_depth=3, random_state=0),
}

print(f"{'Model':<35} {'Train Acc':>10} {'Test Acc':>10}")
print("-" * 58)
for name, m in models_tree.items():
    m.fit(X_ctr, y_ctr)
    print(f"{name:<35} {m.score(X_ctr, y_ctr):>10.4f} {m.score(X_cte, y_cte):>10.4f}")


## 2.8 Kernelized Support Vector Machines (SVM)

SVM berusaha menemukan batas keputusan (hyperplane) yang memaksimalkan
margin antara dua kelas.

**Support vectors** adalah titik-titik training yang paling dekat dengan batas keputusan.
Hanya support vectors inilah yang menentukan posisi batas, bukan semua titik training.

**Kernel trick:**
SVM linear hanya bisa membuat batas keputusan linear. Kernel trick
secara implisit memetakan data ke dimensi yang lebih tinggi tanpa benar-benar
menghitung transformasinya, memungkinkan batas keputusan non-linear.

**Kernel yang umum digunakan:**
- `linear`: tidak ada transformasi, batas linear
- `rbf` (Radial Basis Function / Gaussian): paling umum, memetakan ke dimensi tak terhingga
- `poly`: fitur polinomial

**Hiperparameter:**
- `C`: regularisasi. C kecil = margin lebih lebar = lebih toleran terhadap kesalahan.
  C besar = margin lebih sempit = lebih ketat memisahkan data training.
- `gamma` (untuk rbf): mengontrol lebar Gaussian. gamma besar = pengaruh setiap titik
  hanya terasa pada jarak dekat = batas lebih kompleks.

**Penting:** SVM sangat sensitif terhadap skala fitur. Selalu lakukan normalisasi dulu.


In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# SVM memerlukan scaling
scaler = StandardScaler()
X_ctr_sc = scaler.fit_transform(X_ctr)
X_cte_sc = scaler.transform(X_cte)

# RBF SVM
svm_rbf = SVC(kernel="rbf", C=10, gamma=0.1, random_state=0)
svm_rbf.fit(X_ctr_sc, y_ctr)

print("=== RBF SVM pada Breast Cancer (data sudah discale) ===")
print(f"Training accuracy : {svm_rbf.score(X_ctr_sc, y_ctr):.4f}")
print(f"Test accuracy     : {svm_rbf.score(X_cte_sc, y_cte):.4f}")
print(f"Jumlah support vectors: {svm_rbf.n_support_}")
print()

# Pentingnya scaling
svm_no_scale = SVC(kernel="rbf", C=10, gamma=0.1)
svm_no_scale.fit(X_ctr, y_ctr)
print("SVM TANPA scaling:")
print(f"  Training accuracy : {svm_no_scale.score(X_ctr, y_ctr):.4f}")
print(f"  Test accuracy     : {svm_no_scale.score(X_cte, y_cte):.4f}")
print()
print("Perbedaan performa yang besar menunjukkan betapa pentingnya scaling untuk SVM.")


In [ ]:
# Grid pencarian C dan gamma
print("Pengaruh C dan gamma pada RBF SVM (data scaled):")
print(f"{'C':>6} {'gamma':>8} {'Train':>10} {'Test':>10}")
print("-" * 38)
for C in [0.1, 1, 10, 100]:
    for gamma in [0.001, 0.01, 0.1, 1]:
        s = SVC(kernel="rbf", C=C, gamma=gamma)
        s.fit(X_ctr_sc, y_ctr)
        print(f"{C:>6} {gamma:>8} {s.score(X_ctr_sc, y_ctr):>10.4f} "
              f"{s.score(X_cte_sc, y_cte):>10.4f}")
    print()


## 2.9 Neural Networks (Multilayer Perceptrons)

Multilayer Perceptron (MLP) adalah neural network sederhana yang terdiri dari:

- **Input layer**: satu neuron per fitur
- **Hidden layers**: representasi yang dipelajari. Setiap neuron menghitung:
  output = activation(w * input + b)
- **Output layer**: satu neuron per kelas (klasifikasi) atau satu neuron (regresi)

**Fungsi aktivasi non-linear** adalah kunci: tanpa fungsi aktivasi non-linear,
manyaknya layer tidak akan membantu karena komposisi fungsi linear tetap linear.

Fungsi aktivasi yang umum:
- **ReLU** (default): f(x) = max(0, x). Cepat dan bekerja baik di banyak kasus.
- **tanh**: f(x) = (e^x - e^-x) / (e^x + e^-x). Output antara -1 dan 1.

**Backpropagation** adalah algoritma yang menghitung gradient loss terhadap
semua bobot dalam jaringan, digunakan untuk memperbarui bobot.

**Hiperparameter penting:**
- `hidden_layer_sizes`: tuple yang menentukan jumlah layer dan neuron per layer.
  Contoh: (100,) = 1 hidden layer dengan 100 neuron; (100, 50) = 2 hidden layers.
- `alpha`: L2 regularization. Semakin besar, semakin terpangkas bobotnya.
- `activation`: fungsi aktivasi ('relu', 'tanh', 'logistic').

**Penting:** Neural network sangat sensitif terhadap skala fitur. Selalu lakukan normalisasi.


In [ ]:
from sklearn.neural_network import MLPClassifier

# MLP dengan 1 hidden layer
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000,
                     alpha=0.01, random_state=0)
mlp.fit(X_ctr_sc, y_ctr)

print("=== MLP (1 hidden layer, 100 neuron) pada Breast Cancer ===")
print(f"Training accuracy : {mlp.score(X_ctr_sc, y_ctr):.4f}")
print(f"Test accuracy     : {mlp.score(X_cte_sc, y_cte):.4f}")
print(f"Jumlah iterasi    : {mlp.n_iter_}")
print()

# Pengaruh arsitektur
print("Pengaruh arsitektur hidden layer:")
print(f"{'hidden_layer_sizes':<25} {'Train Acc':>10} {'Test Acc':>10}")
print("-" * 48)
for hidden in [(50,), (100,), (200,), (100, 50), (100, 100), (100, 100, 50)]:
    m = MLPClassifier(hidden_layer_sizes=hidden, max_iter=1000,
                       alpha=0.01, random_state=0)
    m.fit(X_ctr_sc, y_ctr)
    print(f"{str(hidden):<25} {m.score(X_ctr_sc, y_ctr):>10.4f} "
          f"{m.score(X_cte_sc, y_cte):>10.4f}")


In [ ]:
# Visualisasi bobot layer pertama MLP
mlp_viz = MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000,
                         alpha=0.01, random_state=0)
mlp_viz.fit(X_ctr_sc, y_ctr)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.ravel()):
    if i < mlp_viz.coefs_[0].shape[1]:
        coef_img = mlp_viz.coefs_[0][:, i]
        # Tampilkan bobot ke-i dari layer pertama sebagai bar
        ax.bar(range(len(coef_img[:20])), coef_img[:20],
               color=["steelblue" if v > 0 else "tomato" for v in coef_img[:20]])
        ax.axhline(0, color="black", linewidth=0.5)
        ax.set_title(f"Neuron {i+1}", fontsize=8)
        ax.set_xticks([])
    else:
        ax.axis("off")

plt.suptitle("Bobot 10 Neuron Pertama di Hidden Layer (20 fitur pertama)", fontsize=10)
plt.tight_layout()
plt.show()


## 2.10 Estimasi Ketidakpastian dari Classifier

Banyak classifier tidak hanya memberikan label prediksi, tapi juga skor kepercayaan
atau probabilitas. Ini sangat berguna untuk:
- Mengetahui seberapa yakin model dengan prediksinya
- Menentukan threshold yang optimal (trade-off precision vs recall)
- Identifikasi sampel yang "meragukan" untuk ditinjau manual

**Dua metode utama:**

**1. `decision_function`**
Mengembalikan skor numerik mentah. Positif berarti condong ke kelas positif,
negatif berarti condong ke kelas negatif. Rentangnya tidak terbatas.
Tidak semua classifier punya method ini.

**2. `predict_proba`**
Mengembalikan probabilitas untuk setiap kelas. Nilainya antara 0 dan 1,
dan jumlah semua kelas = 1. Lebih intuitif bagi end-user.

Tidak semua classifier mendukung kedua method ini.
Naive Bayes, Random Forest, dan Gradient Boosting punya `predict_proba`.
SVM defaultnya hanya punya `decision_function` (butuh `probability=True` untuk predict_proba).


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_demo = GradientBoostingClassifier(n_estimators=100, random_state=0)
gb_demo.fit(X_ctr, y_ctr)

# decision_function
dec_func = gb_demo.decision_function(X_cte)
print("=== decision_function ===")
print(f"Shape output    : {dec_func.shape}")
print(f"5 nilai pertama : {dec_func[:5].round(3)}")
print(f"Nilai min       : {dec_func.min():.3f}")
print(f"Nilai max       : {dec_func.max():.3f}")
print()

# predict_proba
proba = gb_demo.predict_proba(X_cte)
print("=== predict_proba ===")
print(f"Shape output       : {proba.shape}")
print(f"5 baris pertama    :")
print(proba[:5].round(4))
print(f"Jumlah per baris   : {proba[:5].sum(axis=1)}")
print()
print("Kolom 0 = probabilitas kelas 0 (malignant)")
print("Kolom 1 = probabilitas kelas 1 (benign)")


In [ ]:
# Visualisasi distribusi skor kepercayaan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# decision_function
axes[0].hist(dec_func[y_cte == 0], bins=30, alpha=0.6, color="tomato",
              label="Malignant (0)", edgecolor="none")
axes[0].hist(dec_func[y_cte == 1], bins=30, alpha=0.6, color="steelblue",
              label="Benign (1)", edgecolor="none")
axes[0].axvline(0, color="black", linewidth=1.5, linestyle="--", label="Threshold=0")
axes[0].set_xlabel("Skor decision_function")
axes[0].set_ylabel("Frekuensi")
axes[0].set_title("Distribusi Skor decision_function")
axes[0].legend()

# predict_proba (probabilitas kelas positif)
axes[1].hist(proba[y_cte == 0, 1], bins=30, alpha=0.6, color="tomato",
              label="Malignant (0)", edgecolor="none")
axes[1].hist(proba[y_cte == 1, 1], bins=30, alpha=0.6, color="steelblue",
              label="Benign (1)", edgecolor="none")
axes[1].axvline(0.5, color="black", linewidth=1.5, linestyle="--", label="Threshold=0.5")
axes[1].set_xlabel("Probabilitas kelas Benign (predict_proba)")
axes[1].set_ylabel("Frekuensi")
axes[1].set_title("Distribusi Probabilitas predict_proba")
axes[1].legend()

plt.tight_layout()
plt.show()
print("Idealnya, dua distribusi terpisah jauh = model sangat yakin dengan prediksinya.")


## 2.11 Perbandingan Semua Model

Mari kita bandingkan performa semua model yang telah dipelajari pada satu dataset
untuk melihat kelebihan dan kekurangan masing-masing.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Dataset yang sama: Breast Cancer
cancer = load_breast_cancer()
X_ctr, X_cte, y_ctr, y_cte = train_test_split(
    cancer.data, cancer.target, random_state=42, stratify=cancer.target)

all_models = {
    "kNN (k=5)":                KNeighborsClassifier(n_neighbors=5),
    "kNN (k=1)":                KNeighborsClassifier(n_neighbors=1),
    "Logistic Regression":      make_pipeline(StandardScaler(),
                                    LogisticRegression(max_iter=5000, C=1)),
    "Decision Tree (depth=4)":  DecisionTreeClassifier(max_depth=4, random_state=0),
    "Random Forest (100)":      RandomForestClassifier(n_estimators=100,
                                    random_state=0, n_jobs=-1),
    "Gradient Boosting":        GradientBoostingClassifier(n_estimators=100,
                                    learning_rate=0.1, max_depth=3, random_state=0),
    "SVM (RBF)":                make_pipeline(StandardScaler(),
                                    SVC(kernel="rbf", C=10, gamma=0.1)),
    "MLP (100,50)":             make_pipeline(StandardScaler(),
                                    MLPClassifier(hidden_layer_sizes=(100, 50),
                                                   max_iter=1000, alpha=0.01,
                                                   random_state=0)),
    "Naive Bayes":              GaussianNB(),
}

print(f"{'Model':<30} {'Train Acc':>10} {'Test Acc':>10}")
print("-" * 53)
results = {}
for name, model in all_models.items():
    model.fit(X_ctr, y_ctr)
    tr = model.score(X_ctr, y_ctr)
    te = model.score(X_cte, y_cte)
    results[name] = {"train": tr, "test": te}
    print(f"{name:<30} {tr:>10.4f} {te:>10.4f}")


In [ ]:
# Visualisasi perbandingan
results_df = pd.DataFrame(results).T.sort_values("test", ascending=True)

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(results_df))
width = 0.35

bars1 = ax.barh(x - width/2, results_df["train"], width,
                 label="Training Accuracy", color="steelblue", alpha=0.8, edgecolor="black")
bars2 = ax.barh(x + width/2, results_df["test"],  width,
                 label="Test Accuracy",     color="tomato",    alpha=0.8, edgecolor="black")

ax.set_yticks(x)
ax.set_yticklabels(results_df.index, fontsize=10)
ax.set_xlabel("Accuracy")
ax.set_title("Perbandingan Semua Model pada Breast Cancer Dataset")
ax.legend()
ax.set_xlim(0.8, 1.02)
ax.axvline(1.0, color="gray", linewidth=0.8, linestyle="--")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


## 2.12 Ringkasan dan Panduan Pemilihan Algoritma

### Tabel Perbandingan Algoritma

| Algoritma | Kelebihan | Kekurangan | Perlu Scaling? |
|-----------|-----------|------------|----------------|
| kNN | Sederhana, intuitif, tidak ada asumsi | Lambat prediksi, sensitif fitur irrelevant | Ya |
| Linear Models | Cepat, interpretable, bagus untuk data besar | Hanya tangkap batas linear | Direkomendasikan |
| Naive Bayes | Sangat cepat, data sparse, dataset kecil | Asumsi independensi sering dilanggar | Tidak |
| Decision Tree | Interpretable, tidak perlu scaling | Mudah overfitting, tidak stabil | Tidak |
| Random Forest | Robust, variance rendah, powerful | Kurang interpretable, lambat | Tidak |
| Gradient Boosting | Sering terbaik untuk tabular data | Banyak hyperparameter, lambat training | Tidak |
| SVM | Sangat baik di data berdimensi tinggi | Harus scaling, lambat data besar | Ya (wajib) |
| Neural Network | Tangkap pola sangat kompleks | Banyak hyperparameter, perlu scaling | Ya (wajib) |

### Kapan Menggunakan Apa?

1. **Mulai dari model sederhana** sebagai baseline: Logistic Regression atau kNN.

2. **Untuk data tabular umum:**
   - Coba Random Forest terlebih dahulu (robust, tidak perlu banyak tuning).
   - Gradient Boosting untuk performa maksimal.

3. **Untuk data berdimensi sangat tinggi** (teks, genomik): SVM dengan kernel linear
   atau Logistic Regression dengan regularisasi.

4. **Untuk dataset kecil:** Naive Bayes atau Linear Models.

5. **Untuk pola sangat kompleks:** Neural Network, tapi butuh data banyak
   dan tuning yang cermat.

6. **Interpretabilitas penting:** Decision Tree atau Logistic Regression.
